# Visualizing Loss Landscapes

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 04.08 |
| Time | ~75 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/04_Optimization/09_loss_landscape_visualization.ipynb)

---

## 🎯 Learning Objective

Visualize 1D and 2D slices of a neural network's loss landscape using filter normalization, and connect landscape sharpness to generalization.

---

## 📐 Theory

Every optimizer in this module has been analyzed on 2D toy bowls, saddles, and quadratics. This
capstone-style notebook asks the harder question: what does a *real* neural network's loss
surface, with millions of parameters, actually look like, and how do you even visualize
something living in a space that large?

### 1D and 2D slices

You cannot plot a function of a million variables directly, but you can plot a **slice**: pick
a trained weight vector $\mathbf{w}^*$ and one or two directions $\mathbf{d}_1, \mathbf{d}_2$,
then evaluate

$$L(\alpha) = \mathcal{L}(\mathbf{w}^* + \alpha\, \mathbf{d}_1) \qquad\text{or}\qquad
L(\alpha,\beta) = \mathcal{L}(\mathbf{w}^* + \alpha\, \mathbf{d}_1 + \beta\, \mathbf{d}_2)$$

over a grid of $\alpha$ (and $\beta$). This projects the impossible-to-visualize
million-dimensional surface down to a curve or a contour plot you can actually look at — with
the caveat that it's only a 1D or 2D *window* into a vastly higher-dimensional object, not the
whole picture.

### Filter normalization

Naively picking $\mathbf{d}_1,\mathbf{d}_2$ as random Gaussian noise vectors runs into a
problem: different layers of a network have wildly different weight scales (recall the
random-matrix-theory framing in `08.04`), so a fixed-size random perturbation might be
enormous relative to one layer's weights and negligible relative to another's — the resulting
plot would mostly show whichever layer happens to be most perturbation-sensitive, not a
faithful picture of the landscape. **Filter normalization** (Li et al., 2018) fixes this: draw
a random direction for each parameter *tensor* separately, then rescale that tensor's direction
to have the *same norm* as the tensor itself. Every layer's perturbation is now proportional to
its own scale, giving a much more representative slice.

### Sharpness and flat vs. sharp minima

Two minima can have identical training loss but very different *local* shape: a **sharp**
minimum sits in a narrow, steep-walled basin (loss increases quickly as you move away in any
direction); a **flat** minimum sits in a wide, shallow basin (loss barely changes nearby). A
natural way to quantify this: measure how much the loss increases, on average, for small random
perturbations of a fixed size around the trained point — this notebook's `sharpness` function
does exactly that. Flat minima are widely believed (and partially supported by generalization
bounds, previewed in `09.04`) to generalize better: since test data induces a slightly
different loss surface than training data, a model sitting in a wide flat basin is more likely
to still be near a low point on the *test* surface too, whereas a model in a narrow sharp basin
can be thrown off by even a small shift.

### Mode connectivity (brief note)

A related, striking empirical finding: two independently trained minima of the same network
(different random initializations) are often connected by a low-loss *path* through weight
space, even though a straight line between them typically passes through a high-loss barrier.
This suggests the loss landscape's many minima are less like isolated pits and more like
connected valleys — a picture quite different from the "many separate local minima" intuition
you might get from low-dimensional optimization (`04.01`).

### SAM: optimizing for flatness directly

**Sharpness-Aware Minimization (SAM)** modifies training to explicitly seek flat minima: instead
of minimizing $\mathcal{L}(\mathbf{w})$, it minimizes the *worst-case* loss within a small
neighborhood, $\max_{\|\boldsymbol{\epsilon}\|\le\rho} \mathcal{L}(\mathbf{w}+\boldsymbol{\epsilon})$,
approximated cheaply by taking one "ascent" step toward the locally worst direction before
computing the real update. We implement a simplified version of this below and verify it finds
measurably flatter minima than plain training.

---

In [ ]:
# Setup
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

A 2D filter-normalized slice through a trained network's loss surface turns an
impossible-to-see million-dimensional object into a contour plot, with the training minimum
sitting at the center.

In [ ]:
rng = np.random.default_rng(0)
torch.manual_seed(0)
N = 200
X = torch.tensor(rng.uniform(-2, 2, size=(N, 4)), dtype=torch.float32)
true_w = torch.tensor(rng.normal(size=4), dtype=torch.float32)
y = (torch.sin(X @ true_w) + 0.05 * torch.randn(N)).unsqueeze(1)

def make_model():
    torch.manual_seed(1)
    return torch.nn.Sequential(torch.nn.Linear(4, 12), torch.nn.Tanh(), torch.nn.Linear(12, 1))

model = make_model()
opt = torch.optim.Adam(model.parameters(), lr=0.05)
for _ in range(300):
    opt.zero_grad()
    torch.nn.functional.mse_loss(model(X), y).backward()
    opt.step()

def filter_normalized_direction(model, seed):
    g = torch.Generator().manual_seed(seed)
    direction = []
    for p in model.parameters():
        d = torch.randn(p.shape, generator=g)
        d = d * (p.norm() / (d.norm() + 1e-10))  # rescale to THIS tensor's own norm
        direction.append(d)
    return direction

d1 = filter_normalized_direction(model, seed=10)
d2 = filter_normalized_direction(model, seed=11)
base_params = [p.clone() for p in model.parameters()]

def loss_at_offset(alpha, beta):
    with torch.no_grad():
        for p, base, dd1, dd2 in zip(model.parameters(), base_params, d1, d2):
            p.copy_(base + alpha * dd1 + beta * dd2)
        loss = torch.nn.functional.mse_loss(model(X), y).item()
    return loss

alphas = np.linspace(-1, 1, 25)
betas = np.linspace(-1, 1, 25)
Z = np.array([[loss_at_offset(a, b) for b in betas] for a in alphas])
with torch.no_grad():  # restore the model's true trained weights before continuing
    for p, base in zip(model.parameters(), base_params):
        p.copy_(base)

fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(1, 2, 1)
cs = ax1.contourf(alphas, betas, Z.T, levels=25, cmap="viridis")
ax1.plot(0, 0, 'r*', markersize=15, label="trained minimum")
ax1.set_xlabel("alpha (direction 1)"); ax1.set_ylabel("beta (direction 2)")
ax1.set_title("Filter-normalized 2D loss slice")
ax1.legend()
fig.colorbar(cs, ax=ax1)

ax2 = fig.add_subplot(1, 2, 2, projection='3d')
Ag, Bg = np.meshgrid(alphas, betas)
ax2.plot_surface(Ag, Bg, Z.T, cmap="viridis", alpha=0.9)
ax2.set_xlabel("alpha"); ax2.set_ylabel("beta"); ax2.set_zlabel("loss")
ax2.set_title("Same slice, as a 3D surface")
plt.tight_layout()
plt.show()

print(f"Loss at the center (alpha=beta=0): {loss_at_offset(0,0):.5f}")
print(f"Loss grid min: {Z.min():.5f}, max: {Z.max():.5f}")
print("The center sits at (or extremely near) the grid's minimum, as it should -- it IS the")
print("point the optimizer actually converged to.")

## 🐍 Implementation from Scratch

We implement filter-normalized random directions exactly as described, confirm the slice's
center matches the model's actual trained loss, and compute a numeric sharpness estimate.

In [ ]:
def sharpness(model, X, y, n_directions=30, radius=0.1, seed=99):
    """Average loss increase under small filter-normalized perturbations -- a cheap numeric
    proxy for 'how curved is the basin around this point'."""
    g = torch.Generator().manual_seed(seed)
    base_loss = torch.nn.functional.mse_loss(model(X), y).item()
    base_params = [p.clone() for p in model.parameters()]
    increases = []
    for _ in range(n_directions):
        with torch.no_grad():
            for p, base in zip(model.parameters(), base_params):
                d = torch.randn(p.shape, generator=g)
                d = d * (radius * base.norm() / (d.norm() + 1e-10))
                p.copy_(base + d)
        increases.append(torch.nn.functional.mse_loss(model(X), y).item() - base_loss)
    with torch.no_grad():
        for p, base in zip(model.parameters(), base_params):
            p.copy_(base)
    return float(np.mean(increases)), base_loss

# Verify the slice's center EXACTLY matches the model's own reported loss (sanity check that
# loss_at_offset correctly reconstructs the network's loss at the true trained point)
center_loss = loss_at_offset(0.0, 0.0)
direct_loss = torch.nn.functional.mse_loss(model(X), y).item()
print(f"Slice center loss: {center_loss:.6f}")
print(f"Model's own loss (computed directly): {direct_loss:.6f}")
print(f"Match: {np.isclose(center_loss, direct_loss, atol=1e-5)}")

sharp, base_loss = sharpness(model, X, y)
print(f"\nSharpness (avg loss increase under radius-0.1 perturbation): {sharp:.5f}")
print(f"Base loss: {base_loss:.5f}")
print("A smaller sharpness value means the basin around this minimum is wider/flatter.")

## 🤖 Why This Matters for AI

**Sharpness-Aware Minimization (SAM)** (Foret et al., 2021) turns the flat-vs-sharp intuition
into an actual training algorithm, and has measurably improved generalization on image and
language benchmarks by explicitly biasing training toward flatter regions of the loss surface.
Below, we implement a simplified SAM update, train two networks (plain Adam vs. SAM-like) on
the same task, and directly measure each one's local sharpness using the estimator from the
Implementation section — confirming SAM finds a genuinely flatter minimum, not just a
differently-shaped one, at the well-documented cost of a somewhat higher final training loss.

In [ ]:
def train_sam_like(steps=300, lr=0.05, rho=0.05):
    """A simplified SAM: perturb toward the worst-case NEARBY direction (scaled gradient),
    compute the REAL gradient there, then step from the ORIGINAL point using that gradient."""
    model_sam = make_model()
    opt = torch.optim.Adam(model_sam.parameters(), lr=lr)
    for _ in range(steps):
        opt.zero_grad()
        torch.nn.functional.mse_loss(model_sam(X), y).backward()
        grad_norm = torch.sqrt(sum((p.grad ** 2).sum() for p in model_sam.parameters()))
        eps = [rho * p.grad / (grad_norm + 1e-12) for p in model_sam.parameters()]
        with torch.no_grad():
            for p, e in zip(model_sam.parameters(), eps):
                p.add_(e)          # ascent step: move to the locally worst-case point
        opt.zero_grad()
        torch.nn.functional.mse_loss(model_sam(X), y).backward()  # gradient AT that point
        with torch.no_grad():
            for p, e in zip(model_sam.parameters(), eps):
                p.sub_(e)          # step back before applying the real update
        opt.step()
    return model_sam

model_sam = train_sam_like()
sharp_plain, loss_plain = sharpness(model, X, y)
sharp_sam, loss_sam = sharpness(model_sam, X, y)

print(f"Plain Adam:   final training loss = {loss_plain:.5f}, sharpness = {sharp_plain:.5f}")
print(f"SAM-like:     final training loss = {loss_sam:.5f}, sharpness = {sharp_sam:.5f}")
print(f"\nSAM finds a measurably FLATTER minimum: {sharp_sam < sharp_plain}")
print(f"...at the cost of a higher training loss: {loss_sam > loss_plain}")
print("This is the well-documented SAM tradeoff -- flatness is traded for a small amount of")
print("training-set fit, in exchange for (empirically) better generalization to new data.")

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.bar(["Plain Adam", "SAM-like"], [sharp_plain, sharp_sam], color=["#C44E52", "#4C72B0"])
ax.set_ylabel("sharpness (avg loss increase under perturbation)")
ax.set_title("SAM explicitly trades training loss for a flatter minimum")
plt.show()

## 🏋️ Exercises

### Warm-up
1. In `filter_normalized_direction`, explain (in a sentence) what would go wrong if you drew
   ONE random direction and applied the exact same perturbation to every parameter tensor
   without rescaling per-tensor. Verify your reasoning by comparing per-layer weight norms in
   the trained `model` with `[p.norm().item() for p in model.parameters()]`.

### Practice
2. Increase the network width from `12` to `64` hidden units and re-generate the 2D loss
   landscape plot. Does the landscape near the minimum look qualitatively different (sharper,
   flatter, more/less symmetric)? Report the new grid's min and max loss values.

### Challenge
3. Vary SAM's `rho` parameter (try $0.01, 0.05, 0.2$) and, for each, train a model and measure
   its sharpness and final training loss. Plot sharpness vs. training loss across the three
   values of `rho` (plus the plain-training baseline) and describe the tradeoff curve you observe.

---

## 📚 Further Reading
- Li et al., ["Visualizing the Loss Landscape of Neural Nets"](https://arxiv.org/abs/1712.09913)
- Foret et al., ["Sharpness-Aware Minimization for Efficiently Improving Generalization"](https://arxiv.org/abs/2010.01412) (SAM)
- Garipov et al., ["Loss Surfaces, Mode Connectivity, and Fast Ensembling of DNNs"](https://arxiv.org/abs/1802.10026)

---

*Next notebook: [Entropy and Information](../05_Information_Theory/01_entropy_and_information.ipynb)*